In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge, Lasso
from sklearn.metrics import classification_report, confusion_matrix, r2_score, mean_squared_error, f1_score
from sklearn.datasets import make_classification
import warnings
warnings.filterwarnings("ignore")

In [10]:
X, y = make_classification(n_samples=1000, n_features=10, n_informative=2, n_redundant=8, weights=[0.9,0.1], flip_y=0,
                           random_state=42)
np.unique(y, return_counts=True)

(array([0, 1]), array([900, 100], dtype=int64))

In [12]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)


In [13]:
params = {
    "solver": "lbfg",
    "max_iter":1000,
    "multi_class": "auto",
    "random_state":8888,
}

lr = LogisticRegression()
lr.fit(X_train, y_train)

y_pred = lr.predict(X_test)

report = classification_report(y_test, y_pred)
print(report)

              precision    recall  f1-score   support

           0       0.95      0.97      0.96       270
           1       0.62      0.50      0.56        30

    accuracy                           0.92       300
   macro avg       0.79      0.73      0.76       300
weighted avg       0.91      0.92      0.92       300



In [18]:
report_dict = classification_report(y_test, y_pred, output_dict=True)
report_dict

{'0': {'precision': 0.9456521739130435,
  'recall': 0.9666666666666667,
  'f1-score': 0.956043956043956,
  'support': 270.0},
 '1': {'precision': 0.625,
  'recall': 0.5,
  'f1-score': 0.5555555555555556,
  'support': 30.0},
 'accuracy': 0.92,
 'macro avg': {'precision': 0.7853260869565217,
  'recall': 0.7333333333333334,
  'f1-score': 0.7557997557997558,
  'support': 300.0},
 'weighted avg': {'precision': 0.9135869565217392,
  'recall': 0.92,
  'f1-score': 0.915995115995116,
  'support': 300.0}}

In [14]:
import mlflow

In [19]:
mlflow.set_tracking_uri("http://127.0.0.1:5000/")
mlflow.set_experiment("First Expremiment 2.0")

with mlflow.start_run(run_name="test_run"):
    mlflow.log_params(params)
    mlflow.log_metrics({
        "accuracy": report_dict['accuracy'],
        "recall_class_0": report_dict['0']['recall'],
        "recall_class_1": report_dict['1']['recall'],
        "f1_score_macro":  report_dict['macro avg']['f1-score']
    })
    mlflow.sklearn.log_model(lr, "Logistic Regression")


2026/04/01 08:21:18 INFO mlflow.tracking.fluent: Experiment with name 'First Expremiment 2.0' does not exist. Creating a new experiment.
2026/04/01 08:21:19 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/01 08:21:20 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run test_run at: http://127.0.0.1:5000/#/experiments/3/runs/255032e90b88409a88fe008e26cbfa08
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3


#### Random Forest

In [20]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier()
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)
report = classification_report(y_test, y_pred)
print(report)


              precision    recall  f1-score   support

           0       0.98      1.00      0.99       270
           1       0.96      0.83      0.89        30

    accuracy                           0.98       300
   macro avg       0.97      0.91      0.94       300
weighted avg       0.98      0.98      0.98       300



### XGB

In [21]:
from xgboost import XGBClassifier

xgb= XGBClassifier(use_label_encoder=False, eval_metric='logloss')
xgb.fit(X_train, y_train)
y_pred = xgb.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.98      1.00      0.99       270
           1       0.96      0.80      0.87        30

    accuracy                           0.98       300
   macro avg       0.97      0.90      0.93       300
weighted avg       0.98      0.98      0.98       300



#### SMOTE Tomek

In [24]:
from imblearn.combine import SMOTETomek
smt = SMOTETomek()
X_train_rus, y_train_rus = smt.fit_resample(X_train, y_train)

np.unique(y_train_rus, return_counts=True)

(array([0, 1]), array([614, 614], dtype=int64))

In [23]:
from xgboost import XGBClassifier

xgb= XGBClassifier(use_label_encoder=False, eval_metric='logloss')
xgb.fit(X_train_rus, y_train_rus)
y_pred = xgb.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.98      0.96      0.97       270
           1       0.69      0.83      0.76        30

    accuracy                           0.95       300
   macro avg       0.84      0.90      0.86       300
weighted avg       0.95      0.95      0.95       300



In [25]:
models = [
    (
        "Logistic Regression", 
        {"C": 1, "solver": 'liblinear'},
        LogisticRegression(C=1, solver='liblinear'), 
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "Random Forest", 
        {"n_estimators": 30, "max_depth": 3},
        RandomForestClassifier(n_estimators=30, max_depth=3), 
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "XGBClassifier",
        {"use_label_encoder": False, "eval_metric": 'logloss'},
        XGBClassifier(use_label_encoder=False, eval_metric='logloss'), 
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "XGBClassifier With SMOTE",
        {"use_label_encoder": False, "eval_metric": 'logloss'},
        XGBClassifier(use_label_encoder=False, eval_metric='logloss'), 
        (X_train_rus, y_train_rus),
        (X_test, y_test)
    )
]

In [26]:
reports = []

for model_name, params, model, train_set, test_set in models:
    X_train = train_set[0]
    y_train = train_set[1]
    X_test = test_set[0]
    y_test = test_set[1]

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    report = classification_report(y_test, y_pred, output_dict=True)
    reports.append(report)

In [27]:
reports

[{'0': {'precision': 0.9454545454545454,
   'recall': 0.9629629629629629,
   'f1-score': 0.9541284403669724,
   'support': 270.0},
  '1': {'precision': 0.6,
   'recall': 0.5,
   'f1-score': 0.5454545454545454,
   'support': 30.0},
  'accuracy': 0.9166666666666666,
  'macro avg': {'precision': 0.7727272727272727,
   'recall': 0.7314814814814814,
   'f1-score': 0.749791492910759,
   'support': 300.0},
  'weighted avg': {'precision': 0.9109090909090909,
   'recall': 0.9166666666666666,
   'f1-score': 0.9132610508757297,
   'support': 300.0}},
 {'0': {'precision': 0.96415770609319,
   'recall': 0.9962962962962963,
   'f1-score': 0.9799635701275046,
   'support': 270.0},
  '1': {'precision': 0.9523809523809523,
   'recall': 0.6666666666666666,
   'f1-score': 0.7843137254901961,
   'support': 30.0},
  'accuracy': 0.9633333333333334,
  'macro avg': {'precision': 0.9582693292370712,
   'recall': 0.8314814814814815,
   'f1-score': 0.8821386478088503,
   'support': 300.0},
  'weighted avg': {'pr

In [32]:
mlflow.set_tracking_uri("http://127.0.0.1:5000/")
mlflow.set_experiment("Anomoly Detection 2.0")

for i, element in enumerate(models):
    model_name = element[0]
    model_params = element[1]
    model = element[2]
    report = reports[i]

    with mlflow.start_run(run_name=model_name):
        mlflow.log_param('model_name', model_name)
        mlflow.log_param('params', model_params)
        mlflow.log_metric("accuracy", report['accuracy'])
        mlflow.log_metric("recall_class_0", report['0']['recall'])
        mlflow.log_metric("recall_class_1", report['1']['recall'])
        mlflow.log_metric("f1_score_macro", report['macro avg']['f1-score'])

        if "XGB" in model_name:
            mlflow.xgboost.log_model(model, "model")
        else:
            mlflow.sklearn.log_model(model, "model")
                          
                          
    


2026/04/01 09:19:03 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/01 09:19:03 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/04/01 09:19:08 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Logistic Regression at: http://127.0.0.1:5000/#/experiments/4/runs/714b4e19555747c99f7b181ef0247493
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4


2026/04/01 09:19:09 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/04/01 09:19:14 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Random Forest at: http://127.0.0.1:5000/#/experiments/4/runs/62dafe5f462f4171a9363d7cadb1499a
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4
🏃 View run XGBClassifier at: http://127.0.0.1:5000/#/experiments/4/runs/b78edea8538b4c65bb3b45c6718f9e4f
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4


2026/04/01 09:19:21 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run XGBClassifier With SMOTE at: http://127.0.0.1:5000/#/experiments/4/runs/eb1402778a3d43189708763a855ad2d1
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4


#### Register Model

In [34]:
model_name = "XGB-Smote"
run_id = input("Enter Run ID")
model_uri = f"runs:/{run_id}/model"

result = mlflow.register_model(model_uri, model_name)

Enter Run ID eb1402778a3d43189708763a855ad2d1


Registered model 'XGB-Smote' already exists. Creating a new version of this model...
2026/04/01 09:50:46 WARNING mlflow.tracking._model_registry.fluent: Run with id eb1402778a3d43189708763a855ad2d1 has no artifacts at artifact path 'model', registering model based on models:/m-3ac576b342204ff9acbc47a5b3ad1413 instead
2026/04/01 09:50:46 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: XGB-Smote, version 1
Created version '1' of model 'XGB-Smote'.


#### Load the Model

In [38]:
model_name = "XGB-Smote"
model_version = 1
model_uri = f"models:/{model_name}/{model_version}"

loaded_model = mlflow.xgboost.load_model(model_uri)
y_pred = loaded_model.predict(X_test)
y_pred[:4]

array([0, 0, 0, 0])

In [39]:
dev_model_uri=f"models:/{model_name}@challenger"
prod_model = "anomoly-detection-prod"

client = mlflow.MlflowClient()
client.copy_model_version(src_model_uri=dev_model_uri, dst_name=prod_model)

Successfully registered model 'anomoly-detection-prod'.
Copied version '1' of model 'XGB-Smote' to version '1' of model 'anomoly-detection-prod'.


<ModelVersion: aliases=[], creation_timestamp=1775018432798, current_stage='None', deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description='', last_updated_timestamp=1775018432798, metrics=None, model_id=None, name='anomoly-detection-prod', params=None, run_id='eb1402778a3d43189708763a855ad2d1', run_link='', source='models:/XGB-Smote/1', status='READY', status_message=None, tags={}, user_id='', version='1', workspace='default'>

In [40]:
model_uri = f"models:/{prod_model}@champion"

loaded_model = mlflow.xgboost.load_model(model_uri)
y_pred = loaded_model.predict(X_test)
y_pred[:4]

array([0, 0, 0, 0])